In [2]:
print("Hi")

Hi


In [20]:
import atexit
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ['MEM0_API_KEY']=os.getenv("MEM0_API_KEY")
os.environ['Google_API_Key'] = os.getenv("Google_API_Key")

import atexit
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [5]:
# ! pip install mem0ai

In [9]:
# PROJECT_ROOT = Path(__file__).resolve().parent #for py files
PROJECT_ROOT = Path.cwd() # for jupyter
GROQ_MODEL = "llama-3.1-8b-instant"
from mem0 import Memory

In [14]:
MEM0_CONFIG = {
    "llm": {
        "provider": "groq",
        "config": {
            "model": GROQ_MODEL,
            "temperature": 0.1,
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "mem0_interview_prep",
            "embedding_model_dims": 768,
            "path": str(PROJECT_ROOT / "qdrant_data"),
        },
    },
    "embedder": {
        "provider": "gemini",
        "config": {
            "model": "models/gemini-embedding-001",
            "embedding_dims": 768,
        },
    },
}

In [16]:
SYSTEM_PROMPT = """You are an AI interview prep coach. Use the following memories \
about the candidate to personalize your questions, avoid repeating covered topics, \
and focus on their weak areas and target role.

Candidate memories:
{memories}

Based on this, either (a) ask the next interview question tailored to them, \
or (b) evaluate their answer and give concise feedback. Be specific and concise."""

In [17]:
CHAIN = (
    ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT),
            ("human", "{question}"),
        ]
    )
    | ChatGroq(model=GROQ_MODEL, temperature=0.3)
    | StrOutputParser()
)

In [18]:
try:
    MEMORY = Memory.from_config(MEM0_CONFIG)
    MEMORY_INIT_ERROR = None
except Exception as exc:
    MEMORY = None
    MEMORY_INIT_ERROR = exc

In [19]:
def _format_memories(search_results) -> str:
    items = search_results.get("results", []) if isinstance(search_results, dict) else search_results
    memories = [
        f"- {item['memory']}"
        for item in items
        if isinstance(item, dict) and item.get("memory")
    ]
    return "\n".join(memories) if memories else "(no prior memories yet)"

In [21]:
def _close_memory() -> None:
    if MEMORY is None:
        return

    try:
        MEMORY.vector_store.client.close()
    except Exception:
        pass

    try:
        MEMORY.close()
    except Exception:
        pass


atexit.register(_close_memory)

<function __main__._close_memory() -> None>

In [22]:
def chat(user_id: str, user_message: str) -> str:
    """Fetch relevant memories, call the LLM, then persist the exchange."""
    if MEMORY is None:
        return f"Setup error: could not initialize Mem0. {MEMORY_INIT_ERROR}"

    try:
        search_results = MEMORY.search(query=user_message, user_id=user_id, limit=5)
        reply = CHAIN.invoke(
            {
                "memories": _format_memories(search_results),
                "question": user_message,
            }
        )
        MEMORY.add(
            [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": reply},
            ],
            user_id=user_id,
        )
        return reply
    except Exception as exc:
        return f"Setup error: {exc}"

In [23]:
def get_all_memories(user_id: str):
    if MEMORY is None:
        return {"error": str(MEMORY_INIT_ERROR), "results": []}

    try:
        return MEMORY.get_all(user_id=user_id)
    except Exception as exc:
        return {"error": str(exc), "results": []}